# GATO — live examples

Interactive companion to the committed scripts in [`examples/`](.) and [`examples/paper-figures/`](paper-figures/). The scripts are the canonical, CLI-runnable path; this notebook lets you poke at the same APIs live.

**Run from the repo root** with a kernel that has the GATO deps (e.g. the GRiD `.venv`), after building the `bsqpN64_indy7` / `bsqpN64_iiwa14` modules (see the [root README](../README.md)). Sections 1–3 are fast; the paper figures are heavier — Fig-4 re-plots with **no GPU**, the rest regenerate on the GPU.

In [ ]:
import os, sys
# make the GATO package + the paper-figures helpers importable
sys.path.insert(0, 'python')
sys.path.insert(0, 'examples/paper-figures')
import numpy as np
import matplotlib.pyplot as plt
from gato.common import figure8
from gato.config import (DEFAULT_SOLVER_PARAMS as SP, FIG8_DEFAULT_PARAMS,
                         INDY7_START_CONFIGS)
URDF = 'examples/indy7_description/indy7.urdf'
N, DT = 64, 0.01
print('ready')

## 1. A single solve
Construct one `BSQP` solver and solve a single figure-8-tracking QP (mirrors [`examples/01_single_solve.py`](01_single_solve.py)).

In [ ]:
from gato.interface import BSQP
solver = BSQP(model_path=URDF, batch_size=1, N=N, dt=DT,
              max_sqp_iters=SP['max_sqp_iters'], kkt_tol=SP['kkt_tol'],
              max_pcg_iters=SP['max_pcg_iters'], pcg_tol=SP['pcg_tol'],
              solve_ratio=SP['solve_ratio'], mu=SP['mu'], q_cost=SP['q_cost'],
              qd_cost=SP['qd_cost'], u_cost=SP['u_cost'], N_cost=SP['N_cost'],
              q_lim_cost=SP['q_lim_cost'], vel_lim_cost=SP['vel_lim_cost'],
              ctrl_lim_cost=SP['ctrl_lim_cost'], rho=SP['rho'], plant_type='indy7')
nx, nu = solver.nx, solver.nu
x0 = np.hstack((INDY7_START_CONFIGS['ready'], np.zeros(nx-6))).astype(np.float32)
ref = figure8(DT, **FIG8_DEFAULT_PARAMS)[:6*N].astype(np.float32)
XU = np.zeros((1, N*(nx+nu)-nu), dtype=np.float32); XU[:, :nx] = x0
XU, t_us = solver.solve(x0.reshape(1,-1), ref.reshape(1,-1), XU)
print(f'solve time {t_us/1000:.3f} ms  | sqp_iters {solver.get_stats()["sqp_iters"]}')

## 2. A batched solve (the headline)
Solve M=8 problems in **one** GPU launch, each with a different damping `rho`; see which member converges best (mirrors [`examples/02_batched_solve.py`](02_batched_solve.py)). Try editing `M` / `rho_batch`.

In [ ]:
M = 8
rho_batch = np.power(10, np.linspace(-4, 1, M)).astype(np.float32)
bs = BSQP(model_path=URDF, batch_size=M, N=N, dt=DT, max_sqp_iters=10,
          kkt_tol=SP['kkt_tol'], max_pcg_iters=SP['max_pcg_iters'], pcg_tol=SP['pcg_tol'],
          solve_ratio=SP['solve_ratio'], mu=SP['mu'], q_cost=SP['q_cost'],
          qd_cost=SP['qd_cost'], u_cost=SP['u_cost'], N_cost=SP['N_cost'],
          q_lim_cost=SP['q_lim_cost'], vel_lim_cost=SP['vel_lim_cost'],
          ctrl_lim_cost=SP['ctrl_lim_cost'], rho=SP['rho'], rho_batch=rho_batch,
          adapt_rho=True, plant_type='indy7')
XU = np.zeros((M, N*(nx+nu)-nu), dtype=np.float32); XU[:, :nx] = x0
XU, t_us = bs.solve(np.tile(x0,(M,1)), np.tile(ref,(M,1)), XU)
merits = np.asarray(bs.get_stats()['final_merit']).reshape(-1)
print(f'{M} solves in {t_us/1000:.3f} ms; best member = {int(np.argmin(merits))}')
for i in range(M): print(f'  rho={rho_batch[i]:.1e}  merit={merits[i]:.3f}')

## 3. Closed-loop MPC tracking
Run the `MPC_GATO` figure-8 loop and plot the realized end-effector path (mirrors [`examples/03_mpc_loop.py`](03_mpc_loop.py)).

In [ ]:
import pinocchio as pin
from gato.mpc_controller import MPC_GATO
model, _, _ = pin.buildModelsFromUrdf(URDF, 'examples/indy7_description/')
mpc = MPC_GATO(model, model_path=URDF, N=N, dt=DT, batch_size=1, plant_type='indy7')
fig8 = figure8(DT, **FIG8_DEFAULT_PARAMS)
xs = np.hstack((INDY7_START_CONFIGS['ready'], np.zeros(model.nv)))
_, st = mpc.run_mpc_fig8(xs, fig8, sim_dt=0.001, sim_time=3.0)
ee = np.asarray(st['ee_actual']); ref6 = fig8.reshape(-1,6)
print(f"mean tracking err {np.mean(st['goal_distances_knot0']):.4f} m")
plt.figure(figsize=(5,5))
plt.plot(ref6[:,0], ref6[:,2], 'k--', alpha=0.5, label='reference')
plt.plot(ee[:,0], ee[:,2], color='#00693E', label='GATO')
plt.xlabel('X (m)'); plt.ylabel('Z (m)'); plt.axis('equal'); plt.legend(); plt.show()

## 4. Paper figures
**Fig-4** (CS1 hyperparameter) re-plots from bundled data with **no GPU**:

In [ ]:
import reproduce_fig4_hparam as fig4, pickle
# load the bundled paper data directly (deterministic, no GPU)
with open(fig4.RECOVERED, 'rb') as f:
    agg = pickle.load(f)
fig4.plot(fig4.aggregate_final(agg), 'fig4_hparam_convergence')
from IPython.display import Image
Image('examples/paper-figures/fig4_hparam_convergence.png')

The remaining figures regenerate on the GPU. Smoke them all (tiny subset), or run a single one — see [`paper-figures/README.md`](paper-figures/README.md):
```bash
python examples/paper-figures/make_all.py --quick
python examples/paper-figures/reproduce_fig3_scalability.py
python examples/paper-figures/reproduce_fig5_disturbance.py
```